# Child Anemia in Peru: reproducible ENDES workflow

This notebook runs the same real-data workflow documented in the repository. It is a reproducibility exercise, not a clinical prediction tool. Run the cells in order and do not skip the data retrieval step.

## Before starting

You need access to the project's shared Google Drive DVC folder. Credentials stay in your own Colab session and must never be added to GitHub. If Google blocks DVC's shared OAuth application, use the private service-account route documented in `05_pipeline/README.md` before running the retrieval cell.

In [ ]:
!git clone https://github.com/kedec123/deivhy-torres-vargas-unmsm.git
%cd deivhy-torres-vargas-unmsm
!git status --short

In [ ]:
!pip install -q -r 05_pipeline/requirements.txt
print('Dependencies installed. If Colab asks for a restart, restart the session and rerun from the first cell.')

In [ ]:
!cd 05_pipeline && dvc pull
print('DVC retrieval finished. If Google blocks default authentication, stop here and follow the private service-account setup in 05_pipeline/README.md; do not commit any credential.')

## Validate the analytical CSV

The file contains de-identified records for children aged 6-35 months. It uses the legacy ENDES outcome for a consistent 2019-2024 trend; the updated 2024 field is retained for sensitivity discussion only.

In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path('05_pipeline/data/endes_anemia_children_2019_2024.csv')
if not data_path.exists():
    raise FileNotFoundError('The CSV is missing. Rerun dvc pull and check Drive access.')

df = pd.read_csv(data_path, dtype={'department_code': 'string'})
print(f'Rows: {len(df):,}')
print(f'Years: {sorted(df.survey_year.unique())}')
print(f'Age range: {df.age_months.min():.0f}-{df.age_months.max():.0f} months')
df.head()

In [ ]:
!python 05_pipeline/src/analyze_endes.py
display(pd.read_csv('05_pipeline/docs/analysis_by_year.csv'))

### Models and preprocessing

This is a tabular binary-classification task: `anemia_legacy` is 1 for anemia and 0 for no anemia. The pipeline uses Logistic Regression as an interpretable baseline and Random Forest as a nonlinear comparison. Numeric features use median imputation and standardization; categorical features use most-frequent imputation and one-hot encoding.

In [1]:
import sys

sys.path.insert(0, '05_pipeline/src')
from train import CATEGORICAL_FEATURES, NUMERIC_FEATURES, build_pipeline

print('Problem: binary classification (anemia / no anemia)')
print(f'Numeric features: {NUMERIC_FEATURES}')
print(f'Categorical features: {CATEGORICAL_FEATURES}')
for model_name in ('logistic_regression', 'random_forest'):
    model = build_pipeline(model_name, seed=42).named_steps['model']
    print(f'{model_name}: {model.__class__.__name__}')

Problem: binary classification (anemia / no anemia)
Numeric features: ['age_months', 'mother_education_code', 'wealth_quintile']
Categorical features: ['child_sex_code', 'residence_code', 'department_code', 'survey_year']
logistic_regression: LogisticRegression
random_forest: RandomForestClassifier


## Run exploratory experiments

The next cell runs logistic regression and random forest for five fixed seeds and writes MLflow records. These are exploratory performance checks, not estimates of clinical usefulness.

In [ ]:
!python 05_pipeline/src/run_experiments.py
results = pd.read_csv('05_pipeline/docs/experiment_results.csv')
results

In [ ]:
!python 11_bias_audit/bias_audit.py
print(Path('11_bias_audit/bias_audit_report.md').read_text()[:1200])

## Optional: inspect MLflow

In a local environment, run `mlflow ui --backend-store-uri ./mlruns` and open `http://127.0.0.1:5000`. Colab does not expose that local address directly without a tunnelling method, so the saved CSV and `mlruns/` artefact are the portable evidence in this notebook.